# BigQuery

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/bigquery/bigquery.ipynb)

BigQuery is the analytical heart of GCP: serverless columnar SQL over Colossus storage, billed per byte scanned on-demand or per slot on capacity. This notebook covers what a data engineer needs: how storage and execution actually work, partitioning and clustering, loading and streaming ingestion, cost control, security and access control, scheduling and orchestration, and performance tuning. Repeated generic interview questions have been merged, and deprecated features (table decorators, Data QnA) removed. The *Things to remember* sections near the end compress the numbers and mistakes worth memorizing.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is BigQuery, and how does it fit into the data engineering ecosystem?

BigQuery is a serverless, disaggregated analytics warehouse. Three systems do the work. Colossus stores the data, replicated and encrypted, in a proprietary columnar format called Capacitor. Dremel is the query engine: a tree of a root server, mixers, and leaf nodes that read Capacitor blocks and shuffle intermediate rows between stages. Borg supplies the compute as slots, which are units of CPU, memory, and network on shared machines. Because storage and compute are separate, you can double the data without resizing anything, and you can buy more compute for one project without moving a byte.

The mental model that matters in an interview is that a query is a distributed program, not a table scan on one machine. Dremel parses your SQL, builds a tree of stages, pushes filters and projections down to the leaves, hashes rows for joins and group-aggregates, and streams partial results up the mixers. Every operator consumes slots. A stage that cannot get slots waits. This is why one query can be slow while the data is small, and why ten queries at once can each get slower even though the data never changed.

A worked example. A table holds 2 TB of events. A dashboard runs `SELECT event_name, COUNT(*) FROM fact_events WHERE event_date = CURRENT_DATE() GROUP BY event_name`. If the table is partitioned by `event_date`, one day is about 2 TB / 365, roughly 5.5 GB, and the scan is small; the group-by still needs shuffle and slots. If the table is not partitioned, all 2 TB is read. At an on-demand list price of about 6.25 USD per TiB, the difference is roughly 0.034 USD versus 12.50 USD, and the response time moves by orders of magnitude. The common mistake is to treat these as the same query and blame "the serverless performance".

The runnable cells use [DuckDB](https://duckdb.org) as a local stand-in for the warehouse: the SQL concepts (partition and cluster pruning, columnar scans, `UNNEST`, `QUALIFY`, `MERGE`) carry over, while BigQuery-only DDL stays in `sql` blocks.

## How do partitioning and clustering improve BigQuery performance?

Partitioning divides a table into physical segments, usually one per day. BigQuery keeps the partition id in the table metadata, so a query with a partition filter can ask the metadata layer which segments to read and never open the rest. Clustering sorts rows inside each partition by up to four columns. Each Capacitor block stores min and max values for the clustered columns, so the scan can skip a block without reading it. Partition pruning removes whole files; cluster pruning removes individual blocks. Both reduce bytes scanned, which cuts latency and, on on-demand billing, cost.

```sql
CREATE TABLE `acme-prod.analytics.fact_orders`
(
  order_id    STRING NOT NULL,
  customer_id STRING,
  order_ts    TIMESTAMP,
  order_date  DATE,
  status      STRING,
  amount      NUMERIC(12,2)
)
PARTITION BY order_date
CLUSTER BY customer_id, status
OPTIONS (partition_expiration_days = 1095, description = 'one row per order')
AS
SELECT order_id, customer_id, order_ts, order_date, status, amount
FROM `acme-prod.staging.orders`;
```

Two rules of thumb. Partition on the column that filters appear on in almost every query, which is normally a date or timestamp truncated to day or month. Cluster on the next one to four columns in filter order, and put the most selective first because the sort order is lexicographic. Expect cluster pruning to help most when the clustered column also appears as a filter in the query; clustering does not help a query that only filters on the partition column.

The common mistake is to use too many partitions or to cluster on the wrong column. A `TIMESTAMP` partition creates one partition per microsecond if you do not truncate it, and a `DISTINCT` customer id as a partition key creates millions of tiny segments, each with metadata overhead. The fix is `PARTITION BY DATE(order_ts)` and `CLUSTER BY customer_id`. Also remember that a filter wrapped in a function such as `WHERE DATE(order_ts) = '2026-10-07'` cannot prune a table partitioned on `order_ts`; filter on the raw column instead.

In [ ]:
import pathlib
import tempfile

import duckdb

con = duckdb.connect()
root = pathlib.Path(tempfile.mkdtemp())

# A "table" partitioned by order_date and clustered (sorted) by customer_id, in small row groups (~blocks).
con.execute(f"""
    COPY (
        SELECT 'o-' || i AS order_id,
               'c-' || lpad(CAST(i % 997 AS VARCHAR), 4, '0') AS customer_id,
               DATE '2026-09-01' + CAST(i % 30 AS INTEGER) AS order_date,
               round(10 + i % 97, 2) AS amount
        FROM range(600000) t(i)
        ORDER BY order_date, customer_id
    ) TO '{root}/fact_orders' (FORMAT parquet, PARTITION_BY (order_date), ROW_GROUP_SIZE 2048)
""")

files = sorted(root.rglob("*.parquet"))
day_files = [f for f in files if "order_date=2026-09-15" in str(f)]
# Block-level min/max stats are what clustering exploits: count row groups that could hold the customer.
groups = con.execute(f"""
    SELECT count(*) AS row_groups,
           count(*) FILTER (WHERE stats_min_value <= 'c-0042' AND stats_max_value >= 'c-0042') AS candidates
    FROM parquet_metadata('{day_files[0]}')
    WHERE path_in_schema = 'customer_id'
""").fetchone()
revenue = con.execute(f"""
    SELECT sum(amount) FROM read_parquet('{root}/fact_orders/*/*.parquet', hive_partitioning = true)
    WHERE order_date = '2026-09-15' AND customer_id = 'c-0042'
""").fetchone()[0]

print(f"partitions: {len(files)} files, the date filter opens {len(day_files)}")
print(f"row groups in that partition: {groups[0]}, whose min/max can contain c-0042: {groups[1]}")
print("revenue:", revenue)
assert len(day_files) == 1 and groups[0] >= 8 and groups[1] <= 2 and revenue is not None

## How do you control cost in BigQuery?

BigQuery has three meters. On-demand query billing charges per byte scanned. Capacity billing charges per slot-time, whether you buy commitments, autoscaling reservations, or an edition. Storage charges per byte per month, with a cheaper physical rate and a simpler logical rate that bills uncompressed logical size. Control means attacking all three: scan less (partition, cluster, `SELECT` only needed columns), spend less slot-time (simpler plans, materialized views, caching), and store less (expiration, long-term pricing, physical billing for append-only tables).

The single best diagnostic is `INFORMATION_SCHEMA.JOBS_BY_PROJECT`. `total_bytes_billed` tells you what on-demand would charge, `total_slot_ms` tells you how much compute the query consumed, and `cache_hit` tells you whether it was free. Join those to `referenced_tables` to find who is scanning the biggest table.

```sql
SELECT job_id, user_email, statement_type, cache_hit,
       ROUND(total_bytes_billed / POW(1024, 4), 3) AS tib_billed,
       total_slot_ms,
       ARRAY_TO_STRING(ARRAY(SELECT CONCAT(t.project_id, '.', t.dataset_id, '.', t.table_id)
                             FROM UNNEST(referenced_tables) AS t), ', ') AS tables_read
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND statement_type IN ('SELECT', 'INSERT', 'MERGE')
ORDER BY total_bytes_billed DESC
LIMIT 20;
```

Other levers: custom query quotas per project or user to cap runaway users, `maximum_bytes_billed` on a query so it fails instead of costing a fortune, partition expiration to delete old data automatically, and materialized views for dashboards that re-run the same aggregation. The common mistake is to optimize only the query text and ignore the fact that a dashboard refreshing every 30 seconds is a workload problem, not a SQL problem. Exact prices change; verify current rates rather than quoting a stale number.

In [ ]:
# On-demand billing is bytes of the columns you reference: SELECT * pays for every column.
wide = root / "events.parquet"
con.execute(f"""
    COPY (
        SELECT i AS event_id, 'user-' || (i % 5000) AS user_id, 'page_view' AS event_name,
               repeat(md5(CAST(i AS VARCHAR)), 8) AS payload_json, now() AS ingested_at
        FROM range(50000) t(i)
    ) TO '{wide}' (FORMAT parquet)
""")
sizes = dict(con.execute(f"""
    SELECT path_in_schema, sum(total_compressed_size) FROM parquet_metadata('{wide}') GROUP BY 1
""").fetchall())
select_star = sum(sizes.values())
two_columns = sizes["user_id"] + sizes["event_name"]
print({k: f"{v:,} B" for k, v in sizes.items()})
print(f"SELECT * reads {select_star:,} B; SELECT user_id, event_name reads {two_columns:,} B "
      f"({two_columns / select_star:.1%})")
assert two_columns < 0.1 * select_star

## How does BigQuery handle data storage and processing?

Storage and processing are physically separate, and that is the defining property of the system. Data lives in Colossus, Google's distributed file system. It is stored column by column in Capacitor, which applies dictionary and run-length encoding, so a `status` column with five values is tiny on disk, while a JSON blob column is expensive. Processing is Dremel, which runs on Borg and is measured in slots. A query plan is a tree: the root server parses and schedules, mixers combine partial results, and leaves scan blocks. Shuffle sends rows between stages over the Jupiter network.

Because storage is columnar and shared, several behaviors follow. Wide tables are cheap to scan if you project few columns, because unreferenced columns are never read. `SELECT *` is expensive on a wide table for exactly that reason. Compression means your logical table size, the number in the UI, and the physical bytes on disk can differ by a large factor. And because compute is elastic, a slow query is almost never "the disk"; it is either too many bytes read, too little slot capacity, or a plan that materializes enormous intermediate results.

```sql
-- Confirm what was actually read and how much compute it took.
SELECT total_bytes_processed,
       total_bytes_billed,
       total_slot_ms,
       cache_hit,
       shuffle_output_bytes
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE job_id = 'job_abc123';

-- EXPLAIN shows the plan without running the query.
EXPLAIN
SELECT customer_id, SUM(amount) AS revenue
FROM `acme-prod.analytics.fact_orders`
WHERE order_date >= DATE '2026-01-01'
GROUP BY customer_id;
```

A common mistake is to reason about BigQuery like a row store with indexes. There are no user indexes and no manual statistics to update; the optimizer uses the table metadata, partitioning, and clustering. Adding an index is not an option. The levers you own are the physical layout, the SQL shape, and the slot capacity. Another mistake is assuming `total_bytes_processed` equals what you pay: with a cache hit or with capacity billing, that number can be large while the marginal cost is zero or fixed.

## What is the difference between BigQuery and traditional relational databases?

A traditional relational database couples storage and compute on one machine or cluster, uses B-tree indexes and a row-oriented page cache, and is built for short transactions, point lookups, and updates. BigQuery decouples the two, uses columnar blocks with block-level min/max metadata instead of indexes, and is built for large scans and aggregations. A row store reads whole rows; BigQuery reads whole columns. A row store expects many small concurrent statements; BigQuery expects a smaller number of heavy ones, each spread across many workers.

Consequences you should know. `UPDATE` and `DELETE` in BigQuery are not free pointer updates; they rewrite affected files in the background with DML, and heavy mutation of a large table is an anti-pattern. There are no foreign keys enforced and no unique constraints. Point lookups without a filter on a key can be slow because there is no index, which is why key-value stores and Bigtable exist for that pattern. Transactions exist but are limited to multi-statement transactions within a session and are not the design center.

A concrete comparison. `SELECT * FROM orders WHERE order_id = 'o-1'` on a billion-row row store with an index on `order_id` is a few page reads. On BigQuery it scans the `order_id` column across the table, minus any cluster pruning, and can read gigabytes. The common mistake in a migration is to port an OLTP access pattern into BigQuery unchanged. The fix is to keep point lookups in the serving database, and land a copy in BigQuery for analytics.

## How do you load data into BigQuery?

There are four main batch paths. `LOAD DATA` or the `bq load` command ingests files from Cloud Storage or local disk and is free for the load itself. `CREATE TABLE ... AS SELECT` transforms and materializes inside BigQuery. The Storage Write API streams rows with exactly-once semantics for a given stream. External tables query files in place in Cloud Storage without loading, at the cost of slower scans. Streaming inserts, the older `tabledata.insertAll` path, are still used for tiny volumes but have a streaming buffer with limited mutation support.

```sql
LOAD DATA OVERWRITE `acme-prod.analytics.raw_orders`
FROM FILES (
  format = 'PARQUET',
  uris = ['gs://acme-landing/orders/dt=2026-10-07/*.parquet']
);

-- Or transform from an external table in place.
CREATE OR REPLACE TABLE `acme-prod.analytics.fact_orders`
PARTITION BY order_date
CLUSTER BY customer_id
AS
SELECT * FROM `acme-prod.analytics.raw_orders_ext`
WHERE order_date = DATE '2026-10-07';

-- Or query GCS files in place with a BigLake external table.
CREATE EXTERNAL TABLE `acme-prod.raw.orders_biglake`
WITH CONNECTION `acme-prod.us.biglake-conn`
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://acme-lake/orders/*.parquet'],
  max_staleness = INTERVAL 15 MINUTE,
  metadata_cache_mode = 'AUTOMATIC'
);
```

Choose by volume, latency, and mutation needs. A nightly batch belongs on `LOAD DATA` or a `SELECT` from GCS, because it is free and fast when files are columnar and reasonably large, ideally 100 MB to 1 GB each. Sub-second ingestion belongs on the Storage Write API with a committed stream, not on `insertAll`. Data that must stay in GCS for other engines belongs in a BigLake or external table. The common mistake is to stream every row individually with the legacy API and then wonder why the streaming buffer blocks `UPDATE` for up to ninety minutes.

## What are the different data export options in BigQuery?

Export means getting bytes out. `EXPORT DATA` writes a query result to Cloud Storage in CSV, JSON, Parquet, or Avro. `bq extract` does the same for a whole table. The Storage Read API streams Arrow or Avro records directly into a client process over gRPC, which is the fast path for Spark, pandas, and Dataflow. You can also `SELECT` into another BigQuery dataset or into a federated engine. Each path has a different cost: `EXPORT DATA` is billed for the bytes read from BigQuery and the storage written to GCS.

```sql
EXPORT DATA
  OPTIONS (
    uri = 'gs://acme-exports/orders/*.parquet',
    format = 'PARQUET',
    overwrite = true
  )
AS
SELECT order_id, customer_id, order_date, amount
FROM `acme-prod.analytics.fact_orders`
WHERE order_date = DATE '2026-10-06';
```

The common mistake is to export a huge table as one CSV file, or to export without a partition filter. A single-file export is capped around 1 GB for CSV and JSON; use a wildcard URI so BigQuery shards the output, and a columnar format so downstream readers can prune. Also remember that CSV export re-encodes nested types, so `EXPORT DATA` with nested columns and `format='CSV'` fails or flattens awkwardly; use Parquet or Avro instead. And an unbounded `EXPORT DATA` is a billed full scan, so filter first.

## Explain the concept of federated queries in BigQuery.

A federated query reads data that lives outside BigQuery without loading it. `EXTERNAL_QUERY` sends SQL to Cloud SQL or Spanner and pulls the result into BigQuery. External tables point at Cloud Storage files. BigLake and BigQuery Omni tables point at GCS, S3, or Azure Blob. The engine pushes down what it can, but for `EXTERNAL_QUERY` the remote database does the heavy lifting, so the remote instance's capacity and latency become yours. Federated queries are for joining a small operational table to a large analytical one, or for avoiding a copy of data you will migrate later.

```sql
SELECT o.order_id, c.email
FROM `acme-prod.analytics.fact_orders` AS o
JOIN EXTERNAL_QUERY(
  'projects/acme-prod/locations/us/connections/cloudsql-conn',
  'SELECT customer_id, email FROM app.customers'
) AS c
ON o.customer_id = c.customer_id
WHERE o.order_date = DATE '2026-10-07';
```

The common mistake is to use federation as a permanent integration instead of a migration bridge. Each federated query pays remote round-trip latency and can hammer an OLTP database that was never sized for analytics. There is also no BigQuery-side statistics or cluster pruning on the remote side, so the optimizer cannot push a filter the remote engine will not use. Use federation for small dimension lookups and one-off migrations, then land the data in BigQuery and join natively.

## What are the best practices for optimizing query performance in BigQuery?

Optimize in a fixed order so you always know which lever mattered. First reduce bytes read: add partition filters, cluster on the following filter columns, and project only the columns you need. Second reduce shuffle and slot-time: pre-aggregate, avoid exploding joins, replace repeated dashboard aggregations with materialized views, and let BI Engine serve the interactive slice. Third manage contention: separate workloads with reservations, cap bytes with `maximum_bytes_billed`, and use result cache where the same query repeats.

```sql
-- 1. Filter on the partition column, not a function of it.
WHERE order_date = DATE '2026-10-07'          -- prunes
-- WHERE DATE(order_ts) = DATE '2026-10-07'   -- does not prune

-- 2. Select only needed columns from a wide table.
SELECT order_id, amount FROM `acme-prod.analytics.fact_orders`;

-- 3. Fail loudly instead of scanning the world.
SELECT * FROM `acme-prod.analytics.fact_orders`
WHERE order_date = DATE '2026-10-07'
-- add: maximum_bytes_billed via the API or a query setting
```

Measure before and after with `INFORMATION_SCHEMA.JOBS_BY_PROJECT`, comparing `total_bytes_billed`, `total_slot_ms`, and `shuffle_output_bytes`. A change that lowers bytes but raises slot-time can be a wash on capacity pricing. The common mistake is to start by rewriting the SQL with clever tricks while the real problem is a query with no partition filter. Check the scan first, then the plan, then the SQL. And never tune against a warm cache without noticing, because `cache_hit` hides the cost you are trying to remove.

## How does BigQuery handle data security?

Security is layered. Identity comes from Cloud IAM, which grants roles such as `roles/bigquery.dataViewer`, `dataEditor`, `jobUser`, and `bigquery.user` at project, dataset, table, or view level. Fine-grained access uses authorized views, authorized datasets, and row-level security policies, plus column-level security through policy tags from the Data Catalog. Data is encrypted at rest by default with Google-managed keys, and you can supply CMEK keys per dataset. In transit everything is TLS. Audit logs record reads and admin actions.

```sql
-- Row-level security: a filter that is ANDed into every query.
CREATE ROW ACCESS POLICY region_filter
ON `acme-prod.analytics.fact_orders`
GRANT TO ('group:emea-analysts@acme.com')
FILTER USING (region = 'EMEA');

-- Column-level: tag a column, then grant a policy tag role.
ALTER TABLE `acme-prod.analytics.fact_orders`
ALTER COLUMN customer_id SET OPTIONS (
  policy_tags = 'projects/acme-prod/locations/us/taxonomies/123/policyTags/456'
);
```

The common mistake is to grant `roles/bigquery.dataViewer` at the project level so every analyst can read every dataset, then try to restrict with views. A user with dataset-level `dataViewer` can query the base table directly and bypass a view. Use authorized views with the base table locked down, or row access policies, which apply regardless of the query path. Another mistake is confusing `jobUser`, who can run queries and pay for them, with `dataViewer`, who can read the bytes; both are required and should be separated.

## What is the difference between a table and a view in BigQuery?

A table stores bytes. A view stores a query. When you query a view, BigQuery expands its SQL into your query and executes that, reading the underlying tables and billing you for those bytes. A table is materialized, so querying it reads only its own data. Views are for logical abstraction, column masking, and joining a stable name to changing logic; tables are for physical layout, partitioning, clustering, and cost control. A view has no partition metadata of its own, so a filter on a view is passed through to the base table where it can still prune.

```sql
CREATE VIEW `acme-prod.analytics.v_orders_emea` AS
SELECT order_id, customer_id, order_date, amount
FROM `acme-prod.analytics.fact_orders`
WHERE region = 'EMEA';

-- Querying the view reads fact_orders and bills for those bytes.
SELECT order_date, SUM(amount)
FROM `acme-prod.analytics.v_orders_emea`
WHERE order_date = DATE '2026-10-07'
GROUP BY order_date;
```

Views can break pruning. If the view itself contains a non-prunable expression, or if it aggregates the partition column away, the outer filter cannot reach the base table. A view that does `SELECT DATE(order_ts) AS d, ...` forces the outer query to filter on `d`, which the base table cannot prune. Use a view for security and naming, and a materialized view or a physical table when the abstraction also needs to be fast. The common mistake is stacking five views and assuming the optimizer sees through all of them.

## Explain the concept of nested and repeated fields in BigQuery.

BigQuery is fundamentally nested. A `STRUCT` is a record with named fields. An `ARRAY<STRUCT<...>>` is a repeated record, which is how you store a one-to-many relationship inside one row without a join. The wire format, Capacitor, stores the leaf columns of the nested structure separately and records repetition and definition levels, so reading one field of a repeated struct does not read the others. This is why BigQuery can be fast on denormalized data that a row store would choke on.

```sql
-- One row per order, with its line items nested.
CREATE TABLE `acme-prod.analytics.orders_nested` (
  order_id STRING,
  order_date DATE,
  customer STRUCT<id STRING, email STRING>,
  items ARRAY<STRUCT<sku STRING, qty INT64, price NUMERIC>>
)
PARTITION BY order_date;

-- Query the nested array with UNNEST.
SELECT o.order_id, i.sku, i.qty * i.price AS line_total
FROM `acme-prod.analytics.orders_nested` AS o, UNNEST(o.items) AS i
WHERE o.order_date = DATE '2026-10-07';
```

The pattern to prefer is denormalize dimensions into nested records and keep only the large fact tables separate. A nested `items` array avoids a join and lets BigQuery read just the item fields it needs. `SELECT *` on a nested schema is costly because it touches every leaf column. The common mistake is to model a star schema with many tiny lookup joins when the lookup is small and stable; nesting it into the fact removes shuffle entirely. The opposite mistake is to nest an array that grows without bound, such as every event for a user, which makes rows huge.

In [ ]:
con.execute("""
    CREATE TABLE orders_nested AS SELECT * FROM (VALUES
        ('o-1', DATE '2026-10-07', {'id': 'c-1', 'segment': 'enterprise'},
         [{'sku': 'A', 'qty': 2, 'price': 5.0}, {'sku': 'B', 'qty': 1, 'price': 20.0}]),
        ('o-2', DATE '2026-10-07', {'id': 'c-2', 'segment': 'smb'},
         [{'sku': 'A', 'qty': 1, 'price': 5.0}])
    ) t(order_id, order_date, customer, items)
""")
# One row per order; UNNEST explodes the repeated field only when the query needs line-item grain.
lines = con.execute("""
    SELECT o.order_id, i.sku, i.qty * i.price AS line_total
    FROM orders_nested AS o, UNNEST(o.items) AS u(i)
    WHERE o.customer.segment = 'enterprise'
    ORDER BY i.sku
""").fetchall()
print(lines)
assert lines == [("o-1", "A", 10.0), ("o-1", "B", 20.0)]

## How can you schedule and automate jobs in BigQuery?

Scheduled queries are the built-in option: a saved query plus a schedule, run by BigQuery itself under an owner identity. They are right for a single statement or a short script with a dependency chain no more complex than "run after the previous one". For multi-step pipelines with retries, backfills, sensors, and cross-system dependencies, you use Cloud Composer (managed Airflow), Workflows, or an external orchestrator. BigQuery also exposes `INFORMATION_SCHEMA.JOBS` so an orchestrator can poll job state instead of guessing.

```sql
-- Schedule this from the BigQuery UI or API; the SQL is just SQL.
CREATE OR REPLACE TABLE `acme-prod.analytics.fact_orders_1d`
PARTITION BY order_date
CLUSTER BY customer_id
AS
SELECT *
FROM `acme-prod.analytics.fact_orders`
WHERE order_date = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY);
```

The common mistake is to chain ten scheduled queries by time offset. They become an implicit, unmonitored DAG that breaks when one run is late, and backfills are impossible because each query only knows "yesterday". Use scheduled queries for leaf tasks and a real orchestrator for anything with a dependency graph, and make every task idempotent by writing to a partition that it can overwrite. Also set an owner service account, because a scheduled query that runs as a departed employee stops working.

## How does BigQuery handle data ingestion from streaming sources?

There are two streaming paths and they are not equivalent. The legacy `tabledata.insertAll` API buffers rows into a streaming buffer that is queryable within seconds but cannot be `UPDATE`d or `DELETE`d until it drains, up to ninety minutes, and it can duplicate rows on retry. The Storage Write API is the modern path: gRPC, protobuf or Arrow rows, exactly-once semantics for a given stream and offset, and support for `_CHANGE_TYPE` to apply upserts and deletes directly, so a streaming table can be mutated while rows are in flight.

```sql
-- Inspect the streaming buffer and its age.
SELECT COUNT(*) AS buffered_rows
FROM `acme-prod.analytics.events` WITH OFFSET;  -- rows are queryable immediately

SELECT MAX(_PARTITIONTIME) AS newest_partition
FROM `acme-prod.analytics.events`;
```

Design a streaming table by partition and cluster, because every insert writes into the current partition. Use `_CHANGE_TYPE = 'UPSERT'` with the Storage Write API, or a `MERGE` on a staging table, when the source is a change feed. The common mistake is to build a streaming dashboard on `insertAll` and then discover that deduplication is impossible without a full-table `MERGE`, because the duplicates are indistinguishable from real events. The second mistake is to expect streamed rows to appear in a specific partition instantly; they land in the buffer first, so partition-level operations on the current day can behave differently while the buffer is non-empty.

## What are the limitations or constraints of using BigQuery?

```sql
-- DML is real and transactional, just not OLTP-scale
begin transaction;
update `acme-prod.analytics.dim_customer`
set segment = 'enterprise'
where customer_id = 'c-1042';
commit transaction;
```

The constraints follow from the design. There are no indexes, no unique or foreign key enforcement, and no primary keys, so correctness is your job. Row-level mutation is expensive because DML rewrites files, and there are per-table rate limits on DML and load jobs. Concurrency is limited by slot capacity, so a burst of queries queues. Streaming has a buffer with mutation limits. Query results and single-table exports have size ceilings. And there are quotas: concurrent query rate per project, load jobs per table per day, and API request rates.

Practical limits worth knowing. A single query result is bounded, and a result larger than the limit needs `EXPORT DATA` or a destination table. A `MERGE` that touches millions of rows is a heavy scan plus a rewrite, so it is not free. Nested arrays cannot be updated element by element cheaply. `UPDATE` on a partitioned table can be scoped to partitions to reduce cost. The common mistake is to treat BigQuery as a transactional database and build an application that does thousands of single-row `UPDATE`s; that pattern belongs in Spanner or Firestore, with an export to BigQuery for analytics.

## How can you monitor and optimize BigQuery costs?

Monitoring has three levels. Billing export to BigQuery gives you the invoice line by line in `gcp_billing_export_v1_<account>`, with cost, credits, labels, and SKU. `INFORMATION_SCHEMA.JOBS_BY_PROJECT` gives you per-job bytes and slot-time, which is where you find the query that caused the invoice line. Cloud Monitoring gives you slot utilization and job counts as time series for alerting. Optimizing is then attribution first: which project, user, dashboard, or table produced the cost, then a targeted fix.

```sql
-- Top spend by label and SKU from the billing export.
SELECT sku.description, IFNULL(labels.value, 'unlabeled') AS team,
       ROUND(SUM(cost), 2) AS cost_usd
FROM `acme-billing.billing_export.gcp_billing_export_v1_01ABCD`
LEFT JOIN UNNEST(labels) AS labels ON labels.key = 'team'
WHERE usage_start_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY sku.description, team
ORDER BY cost_usd DESC
LIMIT 20;

-- Then find the offending jobs.
SELECT user_email, ROUND(SUM(total_bytes_billed) / POW(1024, 4), 2) AS tib
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY user_email
ORDER BY tib DESC;
```

The common mistake is to add budgets and stop there. A budget alert tells you money was spent but not which query spent it, so it does not reduce cost. The other mistake is to optimize `total_bytes_billed` alone while the project is on capacity billing, where the marginal cost of a cached or small query is already zero and the real lever is keeping autoscaling slots busy or reducing the reservation floor. Match the optimization to the billing model you are actually on.

## Explain the difference between BigQuery slots and slots reservation.

A slot is the unit of computational capacity: roughly a core with associated memory and network that an operator can use. You never address a slot directly; the scheduler assigns work to available slots in your reservation. A reservation is a pool of slots you own in a specific region and edition, with an optional baseline and an optional autoscaling maximum. An assignment attaches a project, folder, or organization to a reservation. On-demand users share a large pool with no reservation at all and are limited by a per-project concurrency heuristic.

```sql
-- Buy a baseline of 500 slots that can autoscale to 2,000.
CREATE RESERVATION `acme-prod`.`us`.`analytics-reservation`
OPTIONS (
  slot_capacity = 500,
  edition = 'ENTERPRISE',
  autoscale_max_slots = 2000
);

CREATE ASSIGNMENT `acme-prod`.`us`.`analytics-reservation`.`analytics-jobs`
OPTIONS (assignee = 'projects/acme-prod');
```

The mental model is a fair-share scheduler inside a fixed pool. If ten queries share a 500-slot reservation, each gets roughly fifty slots and they finish in about the same time as one query would alone at fifty slots. There is no per-query priority knob that preempts others, so isolating a critical workload means giving it its own reservation or assignment. The common mistake is to say slots are CPUs you can hand out one by one; they are a scheduling currency, and the meaningful decisions are reservation size, edition, and which projects are assigned.

## What is the difference between a view and a materialized view in BigQuery?

A view is stored SQL; it costs nothing to keep and reads the base tables every time. A materialized view is a real table that BigQuery incrementally maintains. It stores the aggregated or filtered result and refreshes either automatically when base data changes, on a schedule, or manually. Queries against a materialized view can be answered from the stored result, and the optimizer can also rewrite a query on the base table to use the materialized view when the aggregate matches.

```sql
CREATE MATERIALIZED VIEW `acme-prod.analytics.mv_daily_revenue`
PARTITION BY order_date
CLUSTER BY customer_id
OPTIONS (enable_refresh = true, refresh_interval_minutes = 60)
AS
SELECT order_date, customer_id, SUM(amount) AS revenue, COUNT(*) AS orders
FROM `acme-prod.analytics.fact_orders`
GROUP BY order_date, customer_id;
```

Materialized views have strict shape rules. They support a subset of SQL: aggregations such as `SUM`, `COUNT`, `MIN`, `MAX`, `AVG` (with `SUM` and `COUNT` for exact `AVG`), `GROUP BY`, joins in some cases, and no window functions, no `ORDER BY`, no `UNION ALL` in the general case, and no user-defined functions. They also cost storage and refresh slot-time. The common mistake is to create a materialized view over a query with a `LIMIT` or a window function and get an error, or to assume refresh is free. Refresh reads the changed partitions, so a materialized view is cheapest when its base table is partitioned and only the recent partitions change.

## Can you explain the concept of data sharding in BigQuery?

Sharding means several tables that together hold one logical dataset, normally one table per day or per entity, addressed with a wildcard such as `events_*`. It was common before partitioned tables existed. BigQuery queries them with `_TABLE_SUFFIX`, and each scanned table is billed. Partitioning is the modern replacement because it presents one schema, one table, one pruning mechanism, and one place to apply policy, while sharding multiplies schema versions, permissions, and metadata.

```sql
-- Wildcard over daily shard tables, filtered by suffix.
SELECT COUNT(*) AS events
FROM `acme-prod.raw.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20261001' AND '20261007'
  AND event_name = 'purchase';   -- note: this column filter still reads each matching shard
```

The common mistake is to shard by day and then filter on a normal column, not on `_TABLE_SUFFIX`. Without a suffix filter the wildcard reads every matching table. A second mistake is to use `_TABLE_SUFFIX` with a leading wildcard, such as `LIKE '%10'`, which cannot prune. If you inherit a sharded dataset, the migration path is to create one partitioned table and insert the shards into their date partitions, then keep a view with the old name so downstream SQL keeps working. Use sharding only when the per-table unit is genuinely separate, such as per-tenant tables with different access.

## How does BigQuery handle schema changes for large tables?

Adding a column is a metadata-only operation. `ALTER TABLE ... ADD COLUMN` updates the schema instantly and existing rows read back as `NULL` for that column, no rewrite and no cost. Renaming a column is also metadata-only. Dropping a column is metadata-only in the sense that the bytes become unreferenced and are reclaimed later. Changing a column type is restricted: widening an `INT64` to `NUMERIC` or `BIGNUMERIC`, or `NUMERIC` to `BIGNUMERIC`, is allowed in some cases; most other conversions require a new table and a `SELECT CAST`.

```sql
ALTER TABLE `acme-prod.analytics.fact_orders`
  ADD COLUMN channel STRING;

ALTER TABLE `acme-prod.analytics.fact_orders`
  RENAME COLUMN channel TO sales_channel;

-- Widening a numeric type is allowed; changing STRING to INT64 is not.
ALTER TABLE `acme-prod.analytics.fact_orders`
  ALTER COLUMN amount SET DATA TYPE BIGNUMERIC;
```

The common mistake is to assume a schema change requires a rewrite and to schedule downtime for it, when adding a nullable column is instant. The opposite mistake is to rely on `ALTER COLUMN ... SET DATA TYPE` for arbitrary conversions; when the target type is not a widening numeric, BigQuery rejects it and you must build a new table. Also remember that schema changes propagate to the streaming buffer and to destination tables of scheduled queries, so a downstream `SELECT *` can suddenly see a new column, which is itself a reason to avoid `SELECT *`.

## How can you control access and permissions in BigQuery?

Access is IAM plus dataset-scoped grants. At the top, project roles such as `roles/bigquery.user` let a principal run jobs and `roles/bigquery.dataViewer` lets them read data. At the dataset level you can grant the same roles to a principal for just that dataset. For finer control, BigQuery adds authorized views (a view that reads a locked base table), authorized datasets (a dataset whose data can be read by another dataset's views), row access policies, column-level security with policy tags, and dataset-level `default_table_expiration`. All grants are additive; there is no deny rule.

```sql
-- Row-level policy: applies to every query of this table.
CREATE ROW ACCESS POLICY eu_only
ON `acme-prod.analytics.fact_orders`
GRANT TO ('group:eu-analysts@acme.com')
FILTER USING (region = 'EU');

-- Column-level: hide PII behind a policy tag.
ALTER TABLE `acme-prod.analytics.fact_orders`
ALTER COLUMN customer_email SET OPTIONS (
  policy_tags = 'projects/acme-prod/locations/us/taxonomies/1/policyTags/9'
);
```

Row access policies are evaluated per query and add a filter, so they can change pruning if the policy column is not the partition column. They also do not apply to `EXPORT DATA` in every path and do not protect against a user who can read the underlying storage. The common mistake is to grant dataset-level `dataViewer` and then build an authorized view, which does not restrict anything because the user can query the base table directly. Lock the base dataset to the view's service account and grant the view to users.

## Can you explain the concept of query caching in BigQuery?

Cache helps repeated dashboard refreshes. It does not help a query that scans different rows each time.

BigQuery has two caches. The result cache stores the output of a completed query for about 24 hours. If the exact same SQL runs again, and none of the referenced tables changed, the answer is returned from the cache with `cache_hit = true` and no charge and no slot usage. The other layer is the long-term storage cache, which keeps recently read Capacitor blocks warm on the compute side; it does not have a user-visible flag the way Snowflake does, but it makes repeated scans of the same blocks quicker.

Cache hits require deterministic SQL and unchanged inputs. Any non-deterministic function, a reference to a temporary or anonymous table, a `CURRENT_TIMESTAMP()` in the query text, or any DML against a referenced table invalidates or prevents the hit. Cache entries are scoped to the user or service account that ran the query and respect access control, so two users do not share a hit if their permissions differ. The common mistake is to benchmark a tuning change with the cache already warm and conclude nothing changed; force a miss or compare `total_slot_ms` and `total_bytes_billed` from `JOBS_BY_PROJECT` instead.

## Can you explain the concept of streaming inserts in BigQuery?

Streaming inserts are the old `tabledata.insertAll` API: you POST rows, BigQuery acknowledges quickly, and the rows are queryable within seconds. What you get back is a best-effort insert. If the request times out and you retry, the same row can be written twice, and there is an `insertId` you can supply for best-effort deduplication within a short window, but it is not a guarantee. Rows first land in a streaming buffer attached to the partition, which is queryable but not mutable by `UPDATE` or `DELETE` until it drains.

```sql
-- The buffer is invisible as a table but visible as recent rows.
SELECT COUNT(*) AS rows_last_minute
FROM `acme-prod.analytics.events`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 MINUTE);

-- Deduplicate after the buffer drains, using a natural key.
SELECT event_id, ANY_VALUE(event_ts) AS event_ts
FROM `acme-prod.analytics.events`
GROUP BY event_id;
```

The common mistake is to use streaming inserts for a critical pipeline and then need to update or delete a streamed row immediately, which fails with a buffer-related error. The second mistake is to assume the `insertId` makes writes exactly-once. If you need exactly-once or in-flight mutations, use the Storage Write API with a committed stream and `_CHANGE_TYPE`, or land rows in Pub/Sub and use Dataflow with a deduplication key.

## How does BigQuery handle data deduplication?

BigQuery does not deduplicate for you. There is no unique constraint, so the pipeline owns the rule. The standard pattern is to deduplicate on a natural or synthetic key, keeping one row per key. For batch loads, `ROW_NUMBER() OVER (PARTITION BY key ORDER BY updated_at DESC)` and filter to row one. For incremental corrections, `MERGE` on the key so a match updates and a miss inserts. For append-only event data with retries, carry an idempotency key and deduplicate in a staging step before publishing to the mart.

```sql
-- Keep the latest version of each order, then merge it into the mart.
WITH ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
  FROM `acme-prod.staging.orders_raw`
  WHERE order_date = DATE '2026-10-07'
)
SELECT * EXCEPT (rn)
FROM ranked
WHERE rn = 1;
```

```sql
MERGE `acme-prod.marts.fact_orders` AS t
USING (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
    FROM `acme-prod.staging.orders_delta`
  ) WHERE rn = 1
) AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET
  status = s.status, amount = s.amount, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT ROW;
```

Two gotchas. First, `MERGE` is a full scan of the target unless you restrict the `ON` clause to the affected partitions; on a large table, always add a date predicate to both sides so pruning works. Second, `MERGE` can fail with "must match at most one source row", which means the source itself is not deduplicated; deduplicate the source in the `USING` subquery first. The common mistake is to assume a re-run of a load is harmless. Without an explicit key and rule, every retry adds rows.

In [ ]:
con.execute("""
    CREATE TABLE fact_orders AS SELECT * FROM (VALUES
        ('o-1', 'created', 10.0, TIMESTAMP '2026-10-06 10:00'),
        ('o-2', 'created', 20.0, TIMESTAMP '2026-10-06 11:00')
    ) t(order_id, status, amount, updated_at);
    CREATE TABLE orders_delta AS SELECT * FROM (VALUES
        ('o-1', 'shipped', 10.0, TIMESTAMP '2026-10-07 09:00'),
        ('o-1', 'shipped', 10.0, TIMESTAMP '2026-10-07 09:00'),   -- duplicate delivery
        ('o-2', 'created', 20.0, TIMESTAMP '2026-10-05 08:00'),   -- stale: older than the target row
        ('o-3', 'created', 30.0, TIMESTAMP '2026-10-07 10:00')
    ) t(order_id, status, amount, updated_at);
""")
# Deduplicate the source first (MERGE fails if a target row matches two source rows), then upsert.
con.execute("""
    MERGE INTO fact_orders AS t
    USING (SELECT * FROM orders_delta
           QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1) AS s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN
        UPDATE SET status = s.status, amount = s.amount, updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT (order_id, status, amount, updated_at)
        VALUES (s.order_id, s.status, s.amount, s.updated_at)
""")
rows = con.execute("SELECT order_id, status FROM fact_orders ORDER BY order_id").fetchall()
print(rows)
assert rows == [("o-1", "shipped"), ("o-2", "created"), ("o-3", "created")]

## Can you explain the concept of streaming buffer in BigQuery?

When rows arrive through the legacy streaming API, they are first written to a streaming buffer that is colocated with the target partition. The buffer is queryable immediately: `SELECT` sees the rows, and they count against the table. But the buffer is append-only and opaque. `UPDATE`, `DELETE`, `MERGE` on buffered rows fails, `EXPORT` and `tabledata.list` may not include them, and partition-level metadata such as `INFORMATION_SCHEMA.PARTITIONS` can lag until the buffer drains. Draining takes from a few minutes to about ninety minutes.

```sql
-- Rows written in the last minute are queryable even while still buffered.
SELECT COUNT(*) AS rows_last_minute
FROM `acme-prod.analytics.events`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 MINUTE);

-- DML against buffered rows can fail until the buffer drains.
DELETE FROM `acme-prod.analytics.events`
WHERE event_date = CURRENT_DATE() AND event_id = 'e-1';
```

The Storage Write API removes most of this problem because committed rows go straight into managed storage, and `_CHANGE_TYPE` lets an upsert or delete apply without waiting for a buffer to drain. The common mistake is to build a streaming table and then run a nightly `MERGE` cleanup against it at a time when the buffer is still holding the day's rows; the `MERGE` cannot touch them and either errors or silently misses them. Wait for the buffer to drain, or move to the Storage Write API.

## How can you optimize data storage costs in BigQuery?

Storage is billed per byte per month, and there are two models. Logical billing charges for the uncompressed logical size of active data and a lower rate for data untouched for 90 days. Physical billing charges for the bytes actually stored, which for a well-encoded columnar table is often far less than logical, and long-term physical is cheaper still. You can set the model per table. On top of that, partition expiration, table expiration, and dropping unused tables remove bytes entirely.

```sql
-- Switch a large, append-only table to physical billing.
ALTER TABLE `acme-prod.analytics.fact_events`
SET OPTIONS (storage_billing_model = 'PHYSICAL');

-- See what each table costs you.
SELECT table_name,
       ROUND(total_logical_bytes / POW(1024, 3), 2) AS logical_gb,
       ROUND(total_physical_bytes / POW(1024, 3), 2) AS physical_gb,
       ROUND(long_term_physical_bytes / POW(1024, 3), 2) AS long_term_physical_gb
FROM `acme-prod.analytics.INFORMATION_SCHEMA.TABLE_STORAGE`
ORDER BY logical_gb DESC;
```

The common mistake is to switch every table to physical billing without checking the ratio. For a table that is rewritten often, or that holds data with poor compression such as UUIDs, physical can be as large as logical and saves nothing, and it removes the predictable logical model. Another mistake is to leave stale staging tables forever. Storage is usually a small part of the bill, so fix the big query scans first, then tidy storage.

## What is the purpose of the INFORMATION_SCHEMA in BigQuery?

`INFORMATION_SCHEMA` is the queryable metadata of one dataset or region. Dataset-scoped views include `TABLES`, `COLUMNS`, `PARTITIONS`, `TABLE_STORAGE`, `VIEWS`, `MATERIALIZED_VIEWS`, and `TABLE_OPTIONS`. Region-scoped views include `JOBS_BY_PROJECT`, `JOBS_BY_USER`, `JOBS_BY_ORGANIZATION`, `JOBS_TIMELINE`, `RESERVATIONS`, and `CAPACITY_COMMITMENTS`. You query them like tables, and the region one is addressed as `region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT` or by project and location.

```sql
SELECT table_name, table_type, creation_time
FROM `acme-prod.analytics.INFORMATION_SCHEMA.TABLES`
WHERE table_name LIKE 'fact_%';

SELECT column_name, data_type, is_nullable
FROM `acme-prod.analytics.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'fact_orders';
```

Two gotchas. First, `JOBS` views are regional, so a job that ran in `EU` is invisible in `region-us`; pick the location that matches the work. Second, `JOBS_BY_PROJECT` retains about 180 days of history and is the source for cost and performance analysis, while `JOBS` under a dataset path gives only jobs in that location up to the same window. The common mistake is to query `INFORMATION_SCHEMA.JOBS` for cost attribution when the project has jobs in several regions, and then wonder why most of them are missing.

## What is the purpose of the BigQuery ML service?

BigQuery ML lets you train and score models with SQL, so the model lives next to the data and no rows leave the warehouse. You `CREATE MODEL` with an algorithm such as linear regression, logistic regression, k-means, boosted trees, or a remote Vertex AI model, then evaluate with `ML.EVALUATE` and predict with `ML.PREDICT`. Training runs as a BigQuery job and consumes slots, and the model is a first-class object in the dataset with its own IAM.

```sql
CREATE OR REPLACE MODEL `acme-prod.ml.churn_model`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['churned'],
  auto_class_weights = TRUE,
  data_split_method = 'AUTO_SPLIT'
) AS
SELECT tenure_months, monthly_spend, support_tickets, churned
FROM `acme-prod.ml.churn_training`;
```

Know where it stops. BigQuery ML is excellent for tabular models sized to a warehouse, and weaker than a dedicated system for deep learning, large-scale hyperparameter search, or serving low-latency predictions from an application. For online serving you export the model or register it in Vertex AI. The common mistake is to train on the raw table without a `TRANSFORM` clause, so the training features and the prediction features are prepared by two different code paths and silently drift.

## How can you monitor and troubleshoot query performance in BigQuery?

Start with `INFORMATION_SCHEMA.JOBS_BY_PROJECT`, which records every job with its timing, bytes, and slot usage. Sort by `total_slot_ms` to find the compute hogs and by `total_bytes_billed` to find the scan hogs. `cache_hit` tells you whether a run was free. `shuffle_output_bytes` tells you how much data moved between stages, which is the signature of a heavy join or group-by. `referenced_tables` tells you what was read, so you can group cost by table.

```sql
SELECT job_id, user_email, statement_type,
       TIMESTAMP_DIFF(end_time, start_time, SECOND) AS elapsed_s,
       total_slot_ms,
       ROUND(total_bytes_billed / POW(1024, 3), 2) AS gb_billed,
       ROUND(shuffle_output_bytes / POW(1024, 3), 2) AS gb_shuffled,
       cache_hit
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
ORDER BY total_slot_ms DESC
LIMIT 20;
```

Then use `EXPLAIN` to see the stage tree without running the query. Look at which stage has the most `records_read` and how long each stage waited. A stage with huge records and small output is filtering or aggregating, which is fine. A stage with huge records in and huge records out is moving everything, which is where shuffle comes from. The common mistake is to look only at wall-clock duration. A query can be slow because it waited for slots, not because it was expensive, and `total_slot_ms` versus elapsed time separates those two cases.

## How does BigQuery handle query optimization and query execution?

A query is compiled into a tree of stages. The leaves scan Capacitor blocks, usually in parallel across many slots. Intermediate stages shuffle rows by join key or group key over the Jupiter network. Mixers combine partial results, and the root returns the answer. The optimizer pushes filters and column projections to the leaves, chooses join order and join strategy, pre-aggregates where it can, and decides how many workers each stage gets. Statistics come from table metadata and from exact counts for small tables.

```sql
EXPLAIN
SELECT c.region, SUM(o.amount) AS revenue
FROM `acme-prod.analytics.fact_orders` AS o
JOIN `acme-prod.analytics.dim_customer` AS c
  ON o.customer_id = c.customer_id
WHERE o.order_date >= DATE '2026-10-01'
  AND c.region = 'EMEA'
GROUP BY c.region;
```

The optimizer cannot fix a bad predicate. A filter that calls a function on the partition column blocks pruning. A join that multiplies rows before a filter moves more data than needed. A `SELECT *` reads every leaf. The common mistake is to assume BigQuery "just optimizes it". The strongest levers are still physical: partition the table on the filter column, cluster the next columns, project the columns you need, and shape joins so the small side is broadcast rather than shuffled.

## Can you explain the concept of wildcard tables in BigQuery?

A wildcard table is a way to query many tables as one. The name has a `*` where the variable part is, such as `acme-prod.raw.events_*`, and `_TABLE_SUFFIX` is a pseudo-column holding the matched part of each table name. A filter on `_TABLE_SUFFIX` lets BigQuery skip whole tables before reading them, and a filter on a normal column does not. Wildcards were how people handled daily tables before partitioned tables existed, and they still matter for sharded exports and per-tenant tables.

```sql
SELECT COUNT(*) AS purchases
FROM `acme-prod.raw.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20260901' AND '20260930'
  AND event_name = 'purchase';
```

The gotcha is that `_TABLE_SUFFIX` is a string, so date comparisons are lexicographic and only work if the suffix format sorts correctly, such as `YYYYMMDD`. A leading wildcard such as `events_*_us` cannot prune on the middle part, and the schema of every matched table must be compatible or the query fails with a schema-mismatch error. The common mistake is to run a wildcard with no suffix filter, which reads every matching table and bills for all of them, exactly like a full scan. If the shards are really one logical table, migrate them to a partitioned table.

## What are the best practices for data modeling in BigQuery?

Model for scans, not for joins. Denormalize small dimensions into nested `STRUCT`s so a fact row carries what it needs without a join. Keep large entities separate and join them on a partition or cluster key when you must. Partition every large fact table by the event date, cluster by the next one to three filter columns, and avoid `SELECT *` in views so projection pushdown survives. Keep raw, staging, and mart datasets separate so access and cost are attributable.

```sql
-- A modeled fact: partitioned, clustered, and dims nested.
CREATE TABLE `acme-prod.marts.fact_orders`
PARTITION BY order_date
CLUSTER BY channel, region
AS
SELECT o.order_id,
       o.order_date,
       o.channel,
       c.region,
       STRUCT(c.customer_id AS id, c.email AS email) AS customer,
       o.amount
FROM `acme-prod.staging.orders` AS o
LEFT JOIN `acme-prod.staging.customers` AS c USING (customer_id);
```

Two anti-patterns to name. The first is a snowflake of many tiny joins, which multiplies shuffle. The second is a single enormous wide table where every query touches many columns and nothing can be pruned. The common mistake is to model for storage efficiency in a system that charges for scans. In BigQuery, some redundancy is a feature: a nested copy of a small dimension removes a join from every query, and repeating a date column in several tables lets each one prune independently.

## How does BigQuery handle data backup and recovery?

You get three mechanisms. Time travel lets you query or restore a table as of any point in the last seven days, and up to the dataset's configured retention window of up to 30 days for managed disaster recovery. `CREATE TABLE ... CLONE` makes a zero-copy clone that shares storage with the source until either diverges, which is the cheap way to snapshot a table before a risky change. And cross-region dataset replication or exports to Cloud Storage give you a copy outside the failure domain.

```sql
-- Snapshot before a risky backfill: cheap, copy-on-write.
CREATE TABLE `acme-prod.backup.fact_orders_20261007`
CLONE `acme-prod.marts.fact_orders`;

-- Recover a table as it was an hour ago.
CREATE OR REPLACE TABLE `acme-prod.marts.fact_orders`
AS SELECT * FROM `acme-prod.marts.fact_orders`
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR);
```

Time travel is not a backup. It lives in the same region and the same account, so a regional outage or an accidental dataset deletion beyond the window takes it with you. Clones share storage, so a clone is not an independent copy either. The common mistake is to treat the seven-day window as durable. For real recovery, export to Cloud Storage in a bucket with its own retention and location, or replicate the dataset to a second region.

## What are the different types of pricing models available for BigQuery?

Three axes. Query pricing is on-demand per byte scanned or capacity through editions and commitments. Storage pricing is logical or physical, each with an active rate and a lower long-term rate after 90 days without modification. Other charges include streaming inserts per row for the legacy API, BI Engine reservation slots, and Data Transfer Service runs. Editions bundle compute, so an edition decision is really a capacity decision with a feature set attached.

```sql
-- Which reservations exist in this region, and how big are they?
SELECT reservation_name, reservation_id, slot_capacity, edition
FROM `region-us`.INFORMATION_SCHEMA.RESERVATIONS;
```

The trade-off is utilization. On-demand is free when idle and expensive when a query scans a lot. Capacity is a fixed cost that rewards steady, high utilization; below roughly half utilization the per-query effective cost can exceed on-demand. Annual commitments discount the per-slot rate but lock you in; monthly commitments cost more and free you sooner. The common mistake is to compare list rates without measuring the actual mix of scan-heavy and scan-light queries. Pull `total_bytes_billed` and `total_slot_ms` from `JOBS_BY_PROJECT` for a month, model both bills, and then decide.

## What is the purpose of the BigQuery Storage API?

The Storage API has two directions. The Storage Read API streams table data to a client over gRPC in Arrow or Avro, with column projection, filter pushdown, and parallel sessions, which is far faster than paging through `tabledata.list`. It is what Spark, Dataflow, and pandas use. The Storage Write API ingests rows over gRPC with exactly-once semantics per stream and offset, and supports `_CHANGE_TYPE` for upserts and deletes, which the legacy streaming API cannot do.

```python
# Storage Read API shape: a client requests a stream, then reads Arrow batches.
# The google-cloud-bigquery-storage package is not installed here.
from google.cloud import bigquery_storage_v1

client = bigquery_storage_v1.BigQueryReadClient()
session = client.create_read_session(
    parent="projects/acme-prod/locations/us",
    read_session=bigquery_storage_v1.types.ReadSession(
        table="projects/acme-prod/datasets/analytics/tables/fact_orders",
        data_format=bigquery_storage_v1.types.DataFormat.ARROW,
        read_options=bigquery_storage_v1.types.ReadSession.TableReadOptions(
            selected_fields=["order_id", "amount"],
            row_restriction="order_date = '2026-10-07'",
        ),
    ),
)
for stream in session.streams:
    for batch in client.read_rows(stream.name):  # one stream per worker
        pass
```

The idea to remember is that the Read API gives one stream per parallel worker, and you are expected to consume them concurrently; reading them sequentially wastes the parallelism. The Write API expects you to pick a stream type: `COMMITTED` for exactly-once, `PENDING` for a two-phase commit across streams, and `BUFFERED` for best-effort low latency. The common mistake is to use the REST streaming API for a high-volume pipeline because it is simpler, then hit per-table throughput limits and duplicate rows on retry.

## How can you handle schema evolution in BigQuery?

Additive change is easy. `ALTER TABLE ... ADD COLUMN` is metadata-only, existing rows read `NULL`, and flexible column names are no longer needed for new fields. Type widening is allowed in limited directions: `INT64` to `NUMERIC`, `BIGNUMERIC`, or `FLOAT64`, and `NUMERIC` to `BIGNUMERIC`. Removing or renaming is also metadata-level, though a rename can break consumers. Incompatible type changes require a new table and a cast. On load, `--autodetect` and `--schema_update_option=ALLOW_FIELD_ADDITION` let new files add columns.

```sql
ALTER TABLE `acme-prod.raw.events`
  ADD COLUMN IF NOT EXISTS ingest_version STRING;

ALTER TABLE `acme-prod.raw.events`
  ALTER COLUMN amount SET DATA TYPE BIGNUMERIC;

-- Incompatible changes need a rebuild.
CREATE OR REPLACE TABLE `acme-prod.raw.events_v2` AS
SELECT * REPLACE (CAST(user_id AS STRING) AS user_id) FROM `acme-prod.raw.events`;
```

The common mistake is to let `SELECT *` in a view or a downstream pipeline absorb every new column, so a source adds a field and a dashboard breaks or a schema check fails unexpectedly. Pin the columns you depend on. The second mistake is to attempt an incompatible `ALTER` on a huge table and discover it is rejected; plan a rebuild with a view alias so consumers keep working. Also add columns as `NULLABLE`, because adding a `REQUIRED` column to a non-empty table is not allowed.

## Can you explain the concept of time travel in BigQuery?

Time travel reads a table as it was. `FOR SYSTEM_TIME AS OF` takes a timestamp or an absolute time expression, and the default window is seven days of history. You can query the past to compare, or copy it into a new table to restore. Datasets also have a configurable `max_time_travel_hours`, up to 168 hours by default and extendable for managed disaster recovery. Clones and snapshots complement this by giving you named restore points instead of a moving window.

```sql
SELECT COUNT(*) AS orders_then
FROM `acme-prod.marts.fact_orders`
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 HOUR);

-- Restore an accidentally emptied table by copying the old version.
CREATE OR REPLACE TABLE `acme-prod.marts.fact_orders` AS
SELECT * FROM `acme-prod.marts.fact_orders`
FOR SYSTEM_TIME AS OF TIMESTAMP '2026-10-07 12:00:00 America/Chicago';
```

Time travel is scoped to the table and the account. It does not survive deleting the dataset, and it is not a substitute for backups outside the region. There is also a subtlety with `CREATE OR REPLACE`: replacing a table preserves its time-travel history, but deleting and recreating it does not. The common mistake is to run `DROP TABLE` as a cleanup step and then discover the history went with it. Use `DELETE` or truncation when you may need to recover.

In [ ]:
# BigQuery keeps a 7-day change history per table. Simulate it with versioned rows and an AS OF filter.
con.execute("""
    CREATE TABLE fact_orders_history AS SELECT * FROM (VALUES
        ('o-1', 10.0, TIMESTAMP '2026-10-01 00:00', TIMESTAMP '2026-10-07 12:30'),
        ('o-1', 12.0, TIMESTAMP '2026-10-07 12:30', NULL),              -- a backfill changed it
        ('o-2', 20.0, TIMESTAMP '2026-10-01 00:00', TIMESTAMP '2026-10-07 13:00')  -- then deleted
    ) t(order_id, amount, valid_from, valid_to)
""")


def revenue_as_of(ts: str) -> float:
    """Like: SELECT SUM(amount) FROM fact_orders FOR SYSTEM_TIME AS OF ts"""
    return con.execute("""
        SELECT coalesce(sum(amount), 0) FROM fact_orders_history
        WHERE valid_from <= CAST(? AS TIMESTAMP)
          AND (valid_to IS NULL OR valid_to > CAST(? AS TIMESTAMP))
    """, [ts, ts]).fetchone()[0]


before, now = revenue_as_of("2026-10-07 12:00"), revenue_as_of("2026-10-07 14:00")
print(f"revenue before the backfill: {before}, now: {now}, delta: {now - before}")
assert (before, now) == (30.0, 12.0)

## How does BigQuery handle data consistency in distributed queries?

BigQuery gives snapshot isolation for a query: a statement sees the state of every table as of the moment the statement started, even while other jobs are loading or merging. Within a multi-statement transaction, BigQuery offers serializable isolation across the statements in that transaction. This means a reader never sees a half-applied load, and a query that joins two tables sees a consistent pair of snapshots. The engine achieves this with metadata versioning rather than locks on the data.

```sql
BEGIN TRANSACTION;

INSERT INTO `acme-prod.marts.dim_customer` (customer_id, region)
VALUES ('c-1042', 'EMEA');

UPDATE `acme-prod.marts.fact_orders`
SET region = 'EMEA'
WHERE customer_id = 'c-1042' AND order_date = DATE '2026-10-07';

COMMIT TRANSACTION;
```

The common mistake is to assume a multi-statement transaction gives you OLTP throughput. BigQuery transactions are intended for atomicity and consistency at the scale of a few statements, not for thousands of small concurrent writers, and they can conflict. The second mistake is to read a table with two separate queries and treat the results as a consistent pair; each query gets its own snapshot, so another job can land between them. Put both reads in one statement or one transaction when consistency across them matters.

## How can you monitor and troubleshoot streaming data pipelines in BigQuery?

Monitor three things: freshness, error rate, and cost. Freshness is the maximum event timestamp per table or per partition; if it stops advancing, the producer or the write stream is stalled. Error rate comes from the write API client's response codes and from the source's dead-letter queue. Cost comes from `INFORMATION_SCHEMA.JOBS_BY_PROJECT` for the queries and from the streaming insert metrics for the write path. Cloud Monitoring exposes write API request counts and errors as time series.

```sql
-- Freshness and volume per partition.
SELECT TIMESTAMP_TRUNC(event_ts, HOUR) AS hour,
       COUNT(*) AS events,
       COUNT(DISTINCT stream_id) AS streams
FROM `acme-prod.analytics.events`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 HOUR)
GROUP BY hour
ORDER BY hour;
```

The common mistake is to alert on total row count instead of freshness. A pipeline that is stuck still has a large count from previous hours, so the count alert never fires while the dashboard quietly serves stale data. Alert on the gap between now and the newest event, and keep a synthetic canary row written every minute so a silent producer is detected even in a low-traffic hour. Track the write stream offset too, because a stream that reconnects with the wrong offset can replay or skip data.

## How does BigQuery handle data skew and hotspots in queries?

Skew is when one worker gets far more rows than the others and everyone waits for it. It shows up as one stage taking much longer than its peers, and as high `shuffle_output_bytes` for a modest input. Classic causes are a join key with a few enormous values, a `GROUP BY` on a low-cardinality column with one dominant group, and an `ARRAY` field that is huge for a few rows. BigQuery does not expose per-worker counts directly, but the stage timings and the ratio of output to input point at it.

```sql
-- Detect a dominant join key before it becomes a skew problem.
SELECT customer_id, COUNT(*) AS rows_per_key
FROM `acme-prod.marts.fact_orders`
WHERE order_date >= DATE '2026-09-01'
GROUP BY customer_id
ORDER BY rows_per_key DESC
LIMIT 20;
```

There is no `DISTRIBUTE BY` hint to fix it directly. The practical remedies are to pre-aggregate before the join so the dominant key carries fewer rows, to salt the key, or to separate the hot partition and union it back. For a `GROUP BY` on a dominant value, adding a second grouping column with some spread helps the shuffle distribute. The common mistake is to blame the data volume when a single skewed key is the bottleneck; the total bytes are fine, but one worker is doing most of the work.

## How do you tune a BigQuery query, step by step?

The workflow is diagnosis before rewrite. Find the expensive job in `INFORMATION_SCHEMA.JOBS_BY_PROJECT`, read the numbers, classify the problem, make one change, then compare the same numbers. A job's `total_bytes_billed` is the scan. `total_slot_ms` is the compute. `shuffle_output_bytes` is how much data moved between stages. `cache_hit` says whether the run was free. Those four columns separate a scan problem from a shuffle problem from a capacity problem before you look at the SQL.

```sql
SELECT job_id, user_email, statement_type, cache_hit,
       ROUND(total_bytes_billed / POW(1024, 4), 3)  AS tib_billed,
       total_slot_ms,
       ROUND(total_slot_ms / 3600000.0, 3)          AS slot_hours,
       ROUND(shuffle_output_bytes / POW(1024, 3), 2) AS gib_shuffled,
       ARRAY_TO_STRING(ARRAY(
         SELECT CONCAT(t.project_id, '.', t.dataset_id, '.', t.table_id)
         FROM UNNEST(referenced_tables) AS t), ', ') AS tables_read
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
  AND statement_type = 'SELECT'
ORDER BY total_slot_ms DESC
LIMIT 25;
```

Map each reading to one lever. A large `tib_billed` with few referenced tables means the filter did not prune, so add a partition filter or clustering. A large `gib_shuffled` with a small scan means a join or group-by is moving rows, so pre-aggregate or broadcast the small side. A large `slot_hours` with a small scan means CPU work in expressions, often on strings or JSON. Zero bytes and `cache_hit = true` means you measured nothing.

The common mistake is to compare two runs by wall-clock time while the second run was served from cache or from a warm slot pool. Compare `total_slot_ms` and `shuffle_output_bytes`, which are still recorded on a cache hit, and re-run with the cache disabled when you need a cold measurement.

## How do you choose partitions and clustering keys?

Partition by day beats cluster by day for a large table because partitioning is metadata pruning at file granularity and clustering is statistical pruning at block granularity. A query for one day against a table partitioned by day opens one segment. The same table clustered by day but not partitioned still reads every partition's metadata and most blocks, because the sort only narrows ranges within each segment. Partition first on the column that appears in almost every filter, then cluster on the next one to four columns in filter order.

```sql
-- Partition on the column every query filters, cluster on the rest.
CREATE TABLE `acme-prod.analytics.fact_events`
PARTITION BY DATE(event_ts)
CLUSTER BY country, device_type, app_id
OPTIONS (partition_expiration_days = 1095)
AS SELECT * FROM `acme-prod.staging.events`;

-- Prunes: filter the raw partition column, not a function of it.
SELECT COUNT(*) FROM `acme-prod.analytics.fact_events`
WHERE event_ts >= TIMESTAMP '2026-10-01 00:00:00'
  AND event_ts <  TIMESTAMP '2026-10-08 00:00:00'
  AND country = 'US';
```

Wildcard and external tables follow the same rule. A wildcard table prunes only on `_TABLE_SUFFIX`, so a filter on a normal column still reads every matching shard. An external table over Hive-partitioned files prunes only when the partition columns are in the path and you filter them. The common mistake is to partition by a `TIMESTAMP` without `DATE()` truncation, which creates a partition per microsecond value and destroys the metadata advantage. The second mistake is to cluster on a near-unique column such as an id; min/max ranges then overlap and prune nothing.

## How do you tune joins, materialized views, BI Engine, and caches?

A join either broadcasts the small side to every worker or shuffles both sides by the join key. Broadcasting is fast because it avoids moving the large table, so keep dimension tables small and filter the large side before the join. An exploding join is one whose output is larger than both inputs, usually because the join key is not unique on both sides. That output then feeds the next stage and the shuffle grows with it.

```sql
-- Pre-aggregate the large side, then join the small dimension.
WITH orders AS (
  SELECT customer_id, SUM(amount) AS revenue
  FROM `acme-prod.marts.fact_orders`
  WHERE order_date >= DATE '2026-10-01'
  GROUP BY customer_id
)
SELECT c.region, SUM(o.revenue) AS revenue
FROM orders AS o
JOIN `acme-prod.marts.dim_customer` AS c USING (customer_id)
GROUP BY c.region;

-- A materialized view serves the repeated dashboard aggregate.
CREATE MATERIALIZED VIEW `acme-prod.marts.mv_region_revenue`
OPTIONS (enable_refresh = true, refresh_interval_minutes = 30)
AS
SELECT order_date, region, SUM(amount) AS revenue
FROM `acme-prod.marts.fact_orders` AS o
JOIN `acme-prod.marts.dim_customer` AS c USING (customer_id)
GROUP BY order_date, region;
```

Then layer the caches. The result cache returns an identical query for free for about 24 hours. BI Engine keeps the interactive slice in memory and reports `bi_engine_statistics`. Materialized views make the repeated aggregate cheap at the source. None of them fix an unpartitioned scan, so use them for repeated shapes, not as the first move.

The common mistake is to join two large tables on a non-unique key and only later notice that the output is ten times the input. Check the row count before and after the join, and confirm that the key you join on is unique where you assume it is.

## How do you choose between on-demand and edition capacity pricing?

On-demand charges per byte scanned and is free when idle. Capacity charges per slot-hour and is free per byte. The break-even is utilization. Multiply the bytes you would scan in a month by the on-demand rate to get the on-demand bill; multiply the slots you would hold continuously by the hours in the month and the slot-hour rate to get the capacity bill. If your sustained slot usage is high and steady, capacity wins; if it is spiky or low, on-demand wins. Autoscaling editions bill only for slots that exist, which shifts the break-even toward capacity for bursty workloads.

```sql
-- Sustained slot usage over the last 30 days, the input to the decision.
SELECT ROUND(SUM(total_slot_ms) / 3600000.0, 1) AS slot_hours_30d,
       ROUND(SUM(total_bytes_billed) / POW(1024, 4), 2) AS tib_billed_30d
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND statement_type = 'SELECT';
```

Add the non-price factors. Capacity gives reservations, workload isolation, and per-reservation attribution; on-demand gives zero commitment and no capacity planning. Editions also gate features such as certain materialized view or BI Engine capabilities, so the decision is not only arithmetic. Prices change, so use the current rate card rather than a remembered number.

The common mistake is to buy a large baseline for a workload that peaks for two hours a day and is idle for the rest. Baseline slots bill continuously. Start from a small baseline with autoscaling, measure actual utilization in `INFORMATION_SCHEMA.RESERVATIONS` and Cloud Monitoring, and only commit once the curve is flat.

## Things to remember: key numbers and limits

Numbers that come up in design reviews. Prices are illustrative us list prices; limits are stable.

| Topic | Number / fact | Why it matters |
|---|---|---|
| On-demand pricing | about 6.25 USD per TiB scanned, 10 MiB minimum per query | `LIMIT` does not reduce bytes scanned; a partition filter does |
| Time travel | 7 days; fail-safe adds 7 more days of recovery | `FOR SYSTEM_TIME AS OF`; snapshots freeze a table longer |
| Clustering | at most 4 clustering columns, ordered most-filtered first | wrong order loses most of the pruning benefit |
| Streaming | the streaming buffer can delay visibility by tens of minutes; legacy `insertAll` is older, Storage Write API is the current path | don't test freshness on just-streamed rows; choose the modern API |
| Partition granularity | thousands of partitions per table; very fine partitions create metadata overhead | daily or hourly for most tables; clustering for the high-cardinality dimension |
| Dry run | free, returns bytes to be processed | dry-run every expensive query shape before production |
| Query cache | 24-hour cache per identical query text over unchanged tables | cached results are free; any table change invalidates |
| Storage billing | logical vs physical bytes differ after compression and time travel | watch `INFORMATION_SCHEMA.TABLE_STORAGE` on churn-heavy tables |

## Things to remember: semantics to memorize

* **Bytes scanned is the only on-demand cost lever.** Partition pruning, clustering, and projecting fewer columns are the controls; `LIMIT` and small result sets do nothing to the bill.
* **Slots vs on-demand is a utilization decision.** Steady, heavy workloads belong on capacity editions with autoscaling slots; bursty ad-hoc belongs on on-demand.
* **Materialized views are automatic rewrites.** The optimizer can answer a query from the MV without changing the SQL — the precomputed-aggregate habit for the warehouse layer.
* **The cache keys on exact query text.** Same logical query with different whitespace or parameters does not hit; standardized query templates do.
* **Streaming rows are not immediately consistent.** The streaming buffer delays availability and excludes those rows from some operations; for immediate analytics micro-batch loads or use Storage Write API with committed mode.
* **Schema changes are metadata operations.** Adding, dropping, and renaming columns does not rewrite data; type narrowing does and needs a rebuild.

## Real world use cases for BigQuery

### 1. Retail: daily sales and inventory marts

A national retailer with 1,200 stores ingests POS transactions (about 4 TB/month, 60 million rows/day) and nightly inventory snapshots. Raw events land in Cloud Storage as Parquet; a `LOAD` job appends into a partitioned fact table; dbt builds a store-day aggregate mart served by Looker.

```sql
create table `acme-prod.analytics.fact_pos_lines`
(
  line_id     string,
  receipt_id  string,
  store_id    string,
  product_id  string,
  sale_date   date,
  qty         int64,
  amount      numeric(12,2)
)
partition by sale_date
cluster by store_id, product_id
options (require_partition_filter = true);

merge `acme-prod.analytics.fact_pos_lines` t
using (
  select * except (rn)
  from (
    select *, row_number() over (partition by line_id order by ingested_at desc) as rn
    from `acme-prod.staging.pos_lines_load`
    where sale_date = date '2026-10-04'
  )
  where rn = 1
) s
on t.line_id = s.line_id
when not matched then insert (line_id, receipt_id, store_id, product_id, sale_date, qty, amount)
values (s.line_id, s.receipt_id, s.store_id, s.product_id, s.sale_date, s.qty, s.amount);
```

Why this design: `sale_date` partitioning keeps dashboard scans to one day of data, `store_id` clustering makes per-store reports prune blocks, and the merge on `line_id` makes reruns idempotent. On-demand pricing plus `require_partition_filter` keeps a 4 TB/month feed cheap.

### 2. Fintech: transaction reconciliation and regulatory reporting

A payments company must reconcile 200 million daily card transactions across a ledger and a processor file, and keep seven years of history for audit. Raw processor files land hourly; a scheduled pipeline reconciles against the internal ledger and writes exceptions to a control table.

```sql
create table `acme-prod.finance.fact_transactions`
(
  txn_id          string,
  account_id      string,
  processor_id    string,
  txn_ts          timestamp,
  txn_date        date,
  amount          numeric(18,2),
  currency        string,
  processor_amount numeric(18,2)
)
partition by txn_date
cluster by processor_id, currency
options (partition_expiration_days = 2555);

create or replace view `acme-prod.finance.v_recon_breaks` as
select txn_id, account_id, txn_date, amount, processor_amount,
       amount - processor_amount as break_amount
from `acme-prod.finance.fact_transactions`
where txn_date = current_date() - 1
  and amount <> processor_amount;
```

Why this design: `numeric` avoids floating-point money errors, date partitioning supports both daily recon and multi-year audit queries, and `partition_expiration_days = 2555` encodes the seven-year retention policy in the table itself. For the immutable regulatory copy, take a snapshot table at month end.

### 3. Marketing: multi-touch attribution across ad platforms

An e-commerce company spends across Google Ads, Meta, and TikTok and wants channel-level ROAS. Ad platform data comes in through Data Transfer Service connectors daily (about 30 GB/day); clickstream and orders are in BigQuery already. A scheduled query joins touchpoints to conversions within a 30-day lookback.

```sql
with touches as (
  select
    user_id,
    channel,
    campaign_id,
    event_ts,
    row_number() over (partition by user_id, channel order by event_ts) as touch_seq
  from `acme-prod.analytics.fact_touchpoints`
  where event_date between date_sub(current_date(), interval 30 day) and current_date()
),
conversions as (
  select user_id, order_id, order_date, amount
  from `acme-prod.analytics.fact_orders`
  where order_date >= date_sub(current_date(), interval 30 day)
)
select
  t.channel,
  count(distinct c.order_id) as attributed_orders,
  sum(c.amount / cnt.touches) as attributed_revenue
from conversions c
join touches t using (user_id)
join (
  select user_id, order_id, count(*) as touches
  from touches join conversions using (user_id)
  group by user_id, order_id
) cnt using (user_id, order_id)
group by t.channel
order by attributed_revenue desc;
```

Why this design: partitioning both tables on the event date keeps the 30-day lookback bounded, and the touchpoint table is clustered by `channel` so per-channel analysis prunes blocks. Attribution logic sits in a materialized view so dashboards stay sub-second.

### 4. SaaS: product analytics with nested events

A B2B SaaS product sends about 500 million events/day, each with a nested `properties` object that differs per event type. The pipeline lands JSON in Cloud Storage, loads into a partitioned table with a `json` column plus a few typed hot columns, and exposes a flattened view for the product team.

```sql
create table `acme-prod.product.events`
(
  event_id   string,
  user_id    string,
  account_id string,
  event_date date,
  event_name string,
  properties json
)
partition by event_date
cluster by account_id, event_name;

create or replace view `acme-prod.product.v_feature_usage` as
select
  event_date,
  account_id,
  event_name,
  json_value(properties, '$.feature') as feature,
  count(*) as events
from `acme-prod.product.events`
group by event_date, account_id, event_name, feature;
```

Why this design: a `json` column absorbs schema churn from hundreds of event types without a migration, while the typed hot columns and clustering keep the common per-account usage queries fast. The view gives analysts a stable interface even as `properties` evolves.

### 5. IoT: fleet telemetry with BI Engine dashboards

A logistics operator receives 12,000 vehicles reporting every 10 seconds (about 100 million messages/day, 2 TB/month) through Pub/Sub. A Dataflow job writes committed streams to a partitioned table; operations dashboards need sub-second refresh on the last 24 hours.

```sql
create table `acme-prod.ops.telemetry`
(
  vehicle_id  string,
  event_ts    timestamp,
  event_date  date,
  lat         float64,
  lon         float64,
  speed_kph   float64,
  fuel_pct    float64
)
partition by event_date
cluster by vehicle_id
options (partition_expiration_days = 365);

-- Hot path: last 24 hours for one fleet region, served from BI Engine
select
  vehicle_id,
  max(event_ts) as last_seen,
  avg(speed_kph) as avg_speed
from `acme-prod.ops.telemetry`
where event_date >= current_date() - 1
  and vehicle_id in (select vehicle_id from `acme-prod.ops.dim_vehicle` where region = 'north')
group by vehicle_id;
```

Why this design: the committed Storage Write API path gives exactly-once ingest without streaming-buffer restrictions, daily partitioning plus `vehicle_id` clustering bounds the 24-hour dashboard scan, and a BI Engine reservation makes the interactive query sub-second. Cold data older than a year expires automatically.

## BigQuery exercises

These use only the standard library; each models a BigQuery decision (partitioning, bytes scanned, slots, pricing, wildcard pruning, job diagnosis). Run each cell and read the asserts.

### Try it 1. Choose partitioning versus clustering

You have a query log. Each entry lists the columns a query filters on. Decide which column to partition by and which to cluster by.

Hint: The date column that appears in every query becomes the partition key. The remaining filter columns become the cluster list, ordered by how often they appear and how selective they are.

Expected result: Partition key `order_date`; cluster keys `customer_id`, `status`, `region` in that order.

In [ ]:
queries = [
    {"filters": ["order_date", "customer_id"]},
    {"filters": ["order_date", "status"]},
    {"filters": ["order_date", "region", "customer_id"]},
]

seen = []
for q in queries:
    for col in q["filters"]:
        if col not in seen:
            seen.append(col)

partition_col = "order_date"
cluster_keys = [c for c in seen if c != partition_col]

assert partition_col == "order_date"
assert cluster_keys == ["customer_id", "status", "region"]
assert len(cluster_keys) <= 4
print("partition:", partition_col)
print("cluster:", ", ".join(cluster_keys))

### Try it 2. Compute bytes scanned per partition

A 2 TiB table is partitioned by day into 730 partitions. A query filters 30 days. Compute the bytes scanned.

Hint: Divide the total by the partition count, then multiply by the number of matching partitions. Real pruning is approximate, so this is an upper bound.

Expected result: About 84 GiB scanned, versus 2 TiB for a full scan.

In [ ]:
total_bytes = 2 * 1024 ** 4          # 2 TiB
partition_count = 730                 # two years of daily partitions
matched_partitions = 30

scanned = total_bytes // partition_count * matched_partitions
full_scan = total_bytes

assert scanned < full_scan
assert 80 < scanned / 1024 ** 3 < 90
print("scanned GiB:", round(scanned / 1024 ** 3, 1))
print("full scan TiB:", full_scan / 1024 ** 4)

### Try it 3. Estimate slot-hours from slot_ms

A job reports `total_slot_ms`. Convert it to slot-hours, then compute the slot-hours a 500-slot batch would consume in 2 minutes.

Hint: One slot-hour is 1000 slot-ms times 3600 seconds. Capacity is slots multiplied by wall-clock seconds.

Expected result: 1.0 slot-hour for the job; about 16.67 slot-hours for the 500-slot 2-minute batch.

In [ ]:
total_slot_ms = 3_600_000
slot_hours = total_slot_ms / 1000 / 3600
assert slot_hours == 1.0

slots = 500
seconds = 120
batch_slot_hours = slots * seconds / 3600
assert round(batch_slot_hours, 2) == 16.67
print("job slot-hours:", slot_hours)
print("batch slot-hours:", round(batch_slot_hours, 2))

### Try it 4. Decide on-demand versus edition

A project scans 120 TiB a month on demand at 6.25 USD per TiB. The alternative is a sustained 400-slot reservation at 0.06 USD per slot-hour, 30 days a month. Which is cheaper?

Hint: Compute the on-demand bill from bytes, and the capacity bill from slots times hours times rate. Compare.

Expected result: On-demand is 750 USD, capacity is 17,280 USD, so on-demand wins at this low utilization.

In [ ]:
usd_per_tib = 6.25
tib_billed = 120
on_demand = tib_billed * usd_per_tib

slots = 400
hours = 30 * 24
usd_per_slot_hour = 0.06
capacity = slots * hours * usd_per_slot_hour

assert on_demand == 750.0
assert abs(capacity - 17280.0) < 1e-9
choice = "on-demand" if on_demand < capacity else "capacity"
assert choice == "on-demand"
print("on-demand USD:", on_demand)
print("capacity USD:", capacity)
print("choice:", choice)

### Try it 5. Compute the cost of a query from bytes billed

A query reports `total_bytes_billed` of 5 TiB. Compute the on-demand cost.

Hint: Divide bytes by 1024 to the fourth power to get TiB, then multiply by the per-TiB rate.

Expected result: 31.25 USD.

In [ ]:
USD_PER_TIB = 6.25

def query_cost_usd(bytes_billed, usd_per_tib=USD_PER_TIB):
    return bytes_billed / 1024 ** 4 * usd_per_tib

bytes_billed = 5 * 1024 ** 4
cost = query_cost_usd(bytes_billed)

assert abs(cost - 31.25) < 1e-9
assert abs(query_cost_usd(0) - 0.0) < 1e-9
print("cost USD:", round(cost, 2))

### Try it 6. Plan a wildcard table filter

Daily shard tables are named `events_YYYYMMDD`. You need 2026-09-15 through 2026-09-21 inclusive.

Hint: `_TABLE_SUFFIX` is a string, so lexical comparison works when the suffix uses zero-padded YYYYMMDD. Inclusive ranges need both ends.

Expected result: Seven shards, `_TABLE_SUFFIX BETWEEN '20260915' AND '20260921'`.

In [ ]:
shards = ["events_%08d" % d for d in range(20260901, 20261001)]
start, end = "20260915", "20260921"

def suffix(shard):
    return shard.split("_")[1]

selected = [s for s in shards if start <= suffix(s) <= end]

assert len(selected) == 7
assert selected[0] == "events_20260915"
assert selected[-1] == "events_20260921"
assert "events_20260914" not in selected
print("suffix filter: _TABLE_SUFFIX BETWEEN '20260915' AND '20260921'")
print("shards scanned:", len(selected))

### Try it 7. Size a clustering key

A query log counts how often each column is filtered. Choose the clustering key list.

Hint: Keep columns filtered often enough to matter, drop near-unique keys, order by frequency, and cap the list at four.

Expected result: `customer_id`, `status`, `region`; `order_id` is dropped as near-unique.

In [ ]:
filter_log = [("customer_id", 1200), ("status", 900), ("region", 400), ("order_id", 30)]

MIN_FILTERS = 100
eligible = [(col, n) for col, n in filter_log if n >= MIN_FILTERS]
cluster_keys = [col for col, _ in sorted(eligible, key=lambda pair: -pair[1])][:4]

assert cluster_keys == ["customer_id", "status", "region"]
assert "order_id" not in cluster_keys
assert len(cluster_keys) <= 4
print("cluster by:", ", ".join(cluster_keys))

### Try it 8. Diagnose a job from JOBS_BY_PROJECT

Two job rows report bytes billed, slot-ms, and shuffle bytes. Classify each as a scan, shuffle, or CPU problem.

Hint: Large bytes with little shuffle is a scan problem. Large shuffle with modest bytes is a join or group-by problem. Large slot-ms with small everything else is CPU in expressions.

Expected result: Job a is `scan`; job b is `shuffle`.

In [ ]:
GIB = 1024 ** 3
TIB = 1024 ** 4

rows = [
    {"job_id": "a", "total_bytes_billed": 2 * TIB, "total_slot_ms": 1_000,
     "shuffle_output_bytes": 0},
    {"job_id": "b", "total_bytes_billed": 1 * GIB, "total_slot_ms": 900_000,
     "shuffle_output_bytes": 50 * GIB},
]

def classify(row):
    if row["total_bytes_billed"] > TIB and row["shuffle_output_bytes"] < GIB:
        return "scan"
    if row["shuffle_output_bytes"] > 10 * GIB:
        return "shuffle"
    return "cpu"

labels = {row["job_id"]: classify(row) for row in rows}
assert labels == {"a": "scan", "b": "shuffle"}
print("a:", labels["a"])
print("b:", labels["b"])